# 🔗 Day 01a: HLD — URL Shortener (TinyURL)

---

## 🎯 What You'll Master Today
- Encoding algorithms (base62, MD5)
- Database schema for read-heavy system
- 301 vs 302 redirects
- Analytics & click tracking
- Scale estimation

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  URL SHORTENER — ARCHITECTURE                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Write Path (create short URL):                                  ║
║  Client ──► API Server ──► Generate short key ──► DB (write)    ║
║                                                                   ║
║  Read Path (redirect):                                           ║
║  Client ──► API Server ──► Cache (Redis) ──► DB (fallback)      ║
║                 │                                                 ║
║                 └── 301/302 redirect to long URL                 ║
║                                                                   ║
║  Full Architecture:                                              ║
║  ┌────────┐   ┌──────┐   ┌───────┐   ┌────────┐               ║
║  │Client  │──►│  LB  │──►│API    │──►│ Redis  │               ║
║  └────────┘   └──────┘   │Server │   │ Cache  │               ║
║                           └───┬───┘   └────────┘               ║
║                               │                                  ║
║                          ┌────▼─────┐                            ║
║                          │ DB       │                            ║
║                          │(NoSQL/SQL)│                           ║
║                          └──────────┘                            ║
║                                                                   ║
║  Scale: Read:Write = 100:1 (read-heavy!)                        ║
║  100M URLs/day → ~1200 writes/sec, ~120K reads/sec             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Back-of-Envelope Estimation
# ============================================================

print("""
📊 SCALE ESTIMATION — TinyURL

Assumptions:
  • 100M new URLs per day
  • Read:Write ratio = 100:1
  • URL average length = 500 bytes
  • Short key = 7 chars
  • Keep for 5 years

QPS:
  Writes: 100M / 86400 ≈ 1200/sec
  Reads:  1200 × 100 = 120,000/sec

Storage (5 years):
  100M/day × 365 × 5 × 500B ≈ 91 TB

Cache (80/20 rule: 20% URLs get 80% traffic):
  Daily read requests = 100M × 100 = 10B
  Cache 20% of daily URLs: 20M × 500B ≈ 10 GB (fits in memory!)

Short key space:
  base62 with 7 chars = 62^7 ≈ 3.5 trillion (way more than 5 years)
""")

In [ ]:
# ============================================================
# 2. Encoding Algorithms
# ============================================================

import hashlib
import string

# --- Approach 1: Base62 Encoding (counter-based) --- 
BASE62 = string.digits + string.ascii_lowercase + string.ascii_uppercase

def base62_encode(num: int) -> str:
    """Convert number to base62 string."""
    if num == 0:
        return BASE62[0]
    result = []
    while num > 0:
        result.append(BASE62[num % 62])
        num //= 62
    return ''.join(reversed(result))

def base62_decode(s: str) -> int:
    """Convert base62 string back to number."""
    num = 0
    for char in s:
        num = num * 62 + BASE62.index(char)
    return num

# Demo counter-based
for counter in [1, 100, 1000000, 999999999]:
    short = base62_encode(counter)
    print(f"ID {counter:>12} → {short:>8} (length={len(short)})")

print()

# --- Approach 2: MD5 Hash (first 7 chars of base62-encoded hash) ---
def hash_url(long_url: str) -> str:
    """Generate short key from hash of URL."""
    md5 = hashlib.md5(long_url.encode()).hexdigest()  # 32 hex chars
    num = int(md5[:12], 16)  # take first 12 hex chars → integer
    return base62_encode(num)[:7]  # take first 7 chars

urls = [
    "https://www.example.com/very-long-article-about-system-design",
    "https://www.hire10x.com/interview-preparation-guide"
]
for url in urls:
    print(f"{url[:50]}... → {hash_url(url)}")

print("""
┌─────────────────────────────────────────────────────────┐
│ Method        │ Pros             │ Cons                 │
├─────────────────────────────────────────────────────────┤
│ Counter+Base62│ No collision     │ Sequential→guessable │
│               │ Short keys       │ Need counter service │
│ MD5 Hash      │ No coordination  │ Collision possible   │
│               │ Stateless        │ Need collision check │
│ UUID          │ Simple           │ Too long (36 chars)  │
│ Snowflake ID  │ Distributed      │ Still need base62    │
└─────────────────────────────────────────────────────────┘

💡 Best approach: Counter with range partitioning
   • Each server gets a range (Server1: 1-1M, Server2: 1M-2M...)
   • No collision, no coordination needed
   • Zookeeper/DB to manage range allocation
""")

In [ ]:
# ============================================================
# 3. Database Schema & Read Optimization
# ============================================================

print("""
📦 DATABASE SCHEMA

Table: urls
┌────────────────────────────────────────────────────┐
│ Column       │ Type         │ Notes                │
├────────────────────────────────────────────────────┤
│ short_key    │ VARCHAR(7)   │ PRIMARY KEY          │
│ long_url     │ VARCHAR(2048)│ NOT NULL             │
│ created_at   │ TIMESTAMP    │ DEFAULT NOW()        │
│ expires_at   │ TIMESTAMP    │ NULL = never expires │
│ user_id      │ BIGINT       │ FK → users           │
│ click_count  │ BIGINT       │ DEFAULT 0            │
└────────────────────────────────────────────────────┘

Table: analytics (separate — don't slow down main table)
┌────────────────────────────────────────────────────┐
│ Column       │ Type         │ Notes                │
├────────────────────────────────────────────────────┤
│ click_id     │ BIGINT       │ PRIMARY KEY          │
│ short_key    │ VARCHAR(7)   │ FK → urls            │
│ clicked_at   │ TIMESTAMP    │                      │
│ ip           │ VARCHAR(45)  │ for geo-location     │
│ user_agent   │ VARCHAR(512) │ browser, OS, device  │
│ referer      │ VARCHAR(2048)│ where click came from│
└────────────────────────────────────────────────────┘

Read Optimization:
  1. Redis cache (short_key → long_url) with TTL
  2. Read replicas for DB (master-slave)
  3. Analytics: async write (Kafka → analytics DB)
""")

print("""
🔀 301 vs 302 REDIRECT

  301 (Permanent):
    • Browser caches it → next time skips our server entirely
    • Pro: Reduces server load
    • Con: Can't track clicks (browser won't hit us again)

  302 (Temporary):
    • Browser always hits our server first
    • Pro: Can track every click for analytics
    • Con: Higher server load

  💡 Most URL shorteners use 302 (analytics > performance)
""")

In [ ]:
# ============================================================
# 4. Complete URL Shortener Service
# ============================================================

import threading

class URLShortener:
    def __init__(self, domain="short.ly"):
        self.domain = domain
        self.counter = 100000  # start from a 3-char base62
        self.url_db = {}       # short_key → {long_url, created_at, clicks}
        self.cache = {}        # short_key → long_url (simulate Redis)
        self.lock = threading.Lock()
    
    def shorten(self, long_url: str) -> str:
        with self.lock:
            self.counter += 1
            short_key = base62_encode(self.counter)
        
        self.url_db[short_key] = {
            "long_url": long_url,
            "created_at": "2025-01-01",
            "clicks": 0
        }
        self.cache[short_key] = long_url
        return f"https://{self.domain}/{short_key}"
    
    def redirect(self, short_key: str) -> str:
        # Check cache first (O(1))
        if short_key in self.cache:
            self.url_db[short_key]["clicks"] += 1
            return self.cache[short_key]
        
        # Fallback to DB
        if short_key in self.url_db:
            long_url = self.url_db[short_key]["long_url"]
            self.cache[short_key] = long_url   # populate cache
            self.url_db[short_key]["clicks"] += 1
            return long_url
        
        return None  # 404
    
    def get_stats(self, short_key: str) -> dict:
        return self.url_db.get(short_key)

# --- Demo ---
service = URLShortener()
urls_to_shorten = [
    "https://www.hire10x.com/system-design-interview-guide",
    "https://www.example.com/very/long/url/that/nobody/likes",
    "https://docs.python.org/3/library/collections.html"
]

for url in urls_to_shorten:
    short = service.shorten(url)
    print(f"{url[:50]:50s} → {short}")

# Redirect simulation
print("\n--- Redirect ---")
key = base62_encode(100001)
for _ in range(3):
    long = service.redirect(key)
    
stats = service.get_stats(key)
print(f"Key '{key}' → {stats['long_url'][:50]}... (clicks={stats['clicks']})")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design TinyURL | Base62 counter, Redis cache, 302 redirect, analytics async |
| 2 | How to generate unique keys? | Counter + base62 (range partition per server). No collision |
| 3 | 301 vs 302? | 301=cached by browser (can't track). 302=always hits server (track clicks) |
| 4 | How to handle collisions? | Counter=no collision. Hash=check DB, rehash with salt |
| 5 | How to scale reads? | Redis cache (20% URLs = 80% traffic), read replicas |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Read-heavy (100:1) → cache aggressively               │
## │  • Base62 counter with range partitioning = best          │
## │  • 302 redirect for analytics, 301 for performance       │
## │  • Analytics → async pipeline (Kafka → analytics DB)     │
## │  • 62^7 ≈ 3.5T keys → enough for decades                │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **HLD — Twitter News Feed**